# M3 step 8: VAE round trip (task 4, RQ2 AI edit)

This simulates an AI tool regenerating a photo. The photo is passed through the image encoder and
decoder of Stable Diffusion (`stabilityai/sd-vae-ft-mse`), which redraws it. The stamp is expected to
break, which is the point: a broken stamp shows the photo was regenerated.

For each stamped photo: resize to 512 by 512, encode to the latent space, decode back, resize to the
original size, save as PNG, read the stamp. The latent mean is used (not a random sample), so the
result is the same on every run.

Two extra checks:

- **Control:** the same resizing to 512 and back without the VAE. This shows that any damage comes
  from the VAE and not from the resizing.
- **Unstamped photos** through the same round trip, to count how many are wrongly read as valid.

Expected: bit accuracy close to 50% and very few valid verdicts.

**GPU:** recommended. About 10 minutes on a T4 for 1,000 round trips (500 stamped, 500 unstamped);
2 to 3 hours on a CPU.

## Setup

**Kaggle:** turn on *Settings > Accelerator > GPU T4 x2* (only notebooks 8, 9 and 11 need it) and
*Internet on*, and attach the `veristamp-m3-inputs` dataset (README.md, "Running on Kaggle"). The first
cell then finds the code, installs what is missing and moves into `m3_experiments`. No path to edit.

**Colab or a local machine:** set the paths once in `settings.py`, then run from the `m3_experiments`
folder.

In [ ]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

In [ ]:
# On Colab instead, uncomment and run once per session:
# from google.colab import drive; drive.mount('/content/drive')
# !git clone https://github.com/KruthikaVirupakshappa/VeriStamp.git /content/VeriStamp
# %cd /content/VeriStamp/m3_experiments
# !pip install -q diffusers transformers accelerate scikit-image lpips

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt

import settings as S
import m3_eval as E
mw = S.setup()                      # imports ../m3_watermark/m3_watermark.py as it is
torch.manual_seed(S.SEED)
np.random.seed(S.SEED)

m3 = mw.load_m3(S.MODEL_PATH)       # load once, reuse for every photo
DEVICE = m3.device
print('Model epoch', m3.epoch, 'on', DEVICE)

In [ ]:
files = E.list_photos(S.TEST_DIR, S.N_TEST_PHOTOS)
if len(files) != S.N_TEST_PHOTOS:
    raise RuntimeError(f'Expected {S.N_TEST_PHOTOS} test photos in {S.TEST_DIR}, found {len(files)}')
# Every photo gets a fixed verdict, P(AI) and tag (E.payload_for), the same in every notebook.
# The stamped PNGs are made once and reused by notebooks 6 to 9.
records = E.stamp_test_set(mw, m3, files, S.WORK_DIR, S.MODEL_PATH, S.SEED)
print(len(records), 'stamped photos in', S.WORK_DIR / 'stamped')
records.head(3)

## Load the VAE

In [ ]:
vae = E.load_vae(DEVICE, S.VAE_NAME)
print('VAE loaded on', next(vae.parameters()).device)

## Run the round trip

In [ ]:
import time
start = time.time()
edits = {'Resize to 512 and back (control)': lambda im, r: E.resize_roundtrip(im, 512),
         'VAE round trip': lambda im, r: E.vae_roundtrip(vae, im, 512)}
per_photo = pd.concat([E.run_edit(mw, m3, records, f, n) for n, f in edits.items()], ignore_index=True)
unstamped = pd.concat([E.run_edit(mw, m3, records, f, n, source='original') for n, f in edits.items()],
                      ignore_index=True)
print(f'{time.time() - start:.0f} s')
summary = E.summarise(per_photo)
false_valid = E.false_valid_summary(unstamped)
display(E.as_percent(summary))
display(E.as_percent(false_valid))

## One example

In [ ]:
r = records.iloc[0]
stamped = E.open_rgb(r['stamped'])
regen = E.vae_roundtrip(vae, stamped)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, im, t in zip(axes, [stamped, regen], ['Stamped', 'After VAE round trip']):
    ax.imshow(im); ax.set_title(t); ax.axis('off')
plt.tight_layout()
(S.RESULTS_DIR / 'figures').mkdir(exist_ok=True)
plt.savefig(S.RESULTS_DIR / 'figures' / 'rq2_vae_example.png', dpi=100, bbox_inches='tight')
plt.show()
print('PSNR of the round trip against the stamped photo:', round(E.psnr(regen, stamped), 2), 'dB')

## Save

In [ ]:
E.save_results(S.RESULTS_DIR, 'rq2_vae', per_photo=pd.concat([per_photo, unstamped]), summary=summary,
               meta={'vae': S.VAE_NAME, 'size': 512, 'latent': 'mode (mean)', 'resize': 'bicubic',
                     'false_valid_unstamped': false_valid.to_dict('records'), 'environment': E.environment_info()})
print(E.to_markdown(E.as_percent(summary)))